# Fine-tune PP-OCRv4 rec head on HVAC tag bubbles

Run on Kaggle T4 (or any single-GPU box, ~30-60 min). Input: `ocr_finetune/` produced locally by `prepare_ocr_finetune.py`. Output: `models/rec_ppocr_v4_hvac/{inference.pdmodel,inference.pdiparams,dict.txt}` ready to drop into the repo.

**Train/inference parity:** the dataset was produced with `ocr_preprocess.preprocess_bubble_crop` (3x upscale + Otsu). `tag_matcher._ocr_bubble_crops_paddle` applies the same preprocessing. Do NOT diverge.

## 0. GPU check

In [ ]:
!nvidia-smi

## 1. Install PaddlePaddle GPU + PaddleOCR (training repo)

In [ ]:
%pip install -q paddlepaddle-gpu==2.5.2 -i https://mirror.baidu.com/pypi/simple
%pip install -q paddleocr==2.7.3 lmdb

In [ ]:
import os, subprocess
if not os.path.exists('PaddleOCR'):
    !git clone -b release/2.7 --depth 1 https://github.com/PaddlePaddle/PaddleOCR.git
%cd PaddleOCR

## 2. Upload the prepared dataset
Upload `ocr_finetune.zip` produced by `prepare_ocr_finetune.py` then unzip.

In [ ]:
# In Kaggle: Add Data → Upload → ocr_finetune.zip
# Then symlink (path varies by upload):
import glob, shutil
candidates = glob.glob('/kaggle/input/**/ocr_finetune.zip', recursive=True) + glob.glob('/content/ocr_finetune.zip')
assert candidates, 'ocr_finetune.zip not found — upload it first'
src = candidates[0]
print('using', src)
shutil.unpack_archive(src, '.')
!ls ocr_finetune | head

## 3. Download PP-OCRv4 English rec pretrained weights

In [ ]:
!mkdir -p pretrain_models
!wget -nc -q https://paddleocr.bj.bcebos.com/PP-OCRv4/english/en_PP-OCRv4_rec_train.tar -O pretrain_models/en_PP-OCRv4_rec_train.tar
!tar -xf pretrain_models/en_PP-OCRv4_rec_train.tar -C pretrain_models/
!ls pretrain_models/en_PP-OCRv4_rec_train

## 4. Write the fine-tune config

In [ ]:
cfg = '''Global:
  debug: false
  use_gpu: true
  epoch_num: 80
  log_smooth_window: 20
  print_batch_step: 20
  save_model_dir: ./output/rec_ppocr_v4_hvac/
  save_epoch_step: 10
  eval_batch_step: [0, 200]
  cal_metric_during_train: true
  pretrained_model: ./pretrain_models/en_PP-OCRv4_rec_train/best_accuracy
  checkpoints:
  save_inference_dir:
  use_visualdl: false
  infer_img:
  character_dict_path: ./ocr_finetune/dict.txt
  max_text_length: 12
  use_space_char: false
  save_res_path: ./output/rec_ppocr_v4_hvac/predicts.txt
Optimizer:
  name: Adam
  beta1: 0.9
  beta2: 0.999
  lr:
    name: Cosine
    learning_rate: 0.0005
    warmup_epoch: 2
  regularizer:
    name: L2
    factor: 3.0e-05
Architecture:
  model_type: rec
  algorithm: SVTR_LCNet
  Transform:
  Backbone:
    name: PPLCNetV3
    scale: 0.95
  Head:
    name: MultiHead
    head_list:
      - CTCHead:
          Neck:
            name: svtr
            dims: 120
            depth: 2
            hidden_dims: 120
            kernel_size: [1, 3]
            use_guide: true
          Head:
            fc_decay: 1.0e-05
      - NRTRHead:
          nrtr_dim: 384
          max_text_length: 12
Loss:
  name: MultiLoss
  loss_config_list:
    - CTCLoss:
    - NRTRLoss:
PostProcess:
  name: CTCLabelDecode
Metric:
  name: RecMetric
  main_indicator: acc
  ignore_space: true
Train:
  dataset:
    name: SimpleDataSet
    data_dir: ./ocr_finetune/
    label_file_list:
      - ./ocr_finetune/train.txt
    transforms:
      - DecodeImage:
          img_mode: BGR
          channel_first: false
      - RecConAug:
          prob: 0.5
          ext_data_num: 2
          image_shape: [48, 320, 3]
          max_text_length: 12
      - RecAug:
      - MultiLabelEncode:
          gtc_encode: NRTRLabelEncode
      - KeepKeys:
          keep_keys: [image, label_ctc, label_gtc, length, valid_ratio]
  loader:
    shuffle: true
    batch_size_per_card: 128
    drop_last: true
    num_workers: 4
Eval:
  dataset:
    name: SimpleDataSet
    data_dir: ./ocr_finetune/
    label_file_list:
      - ./ocr_finetune/val.txt
    transforms:
      - DecodeImage:
          img_mode: BGR
          channel_first: false
      - MultiLabelEncode:
          gtc_encode: NRTRLabelEncode
      - KeepKeys:
          keep_keys: [image, label_ctc, label_gtc, length, valid_ratio]
  loader:
    shuffle: false
    drop_last: false
    batch_size_per_card: 128
    num_workers: 4
'''
open('configs/rec/PP-OCRv4/en_PP-OCRv4_rec_hvac.yml', 'w').write(cfg)
print('config written')

## 5. Train
Expected: 30-60 min on T4. Watch the `acc` metric — target ≥ 0.95 by epoch 30.

In [ ]:
!python tools/train.py -c configs/rec/PP-OCRv4/en_PP-OCRv4_rec_hvac.yml

## 6. Export inference model

In [ ]:
!python tools/export_model.py \
    -c configs/rec/PP-OCRv4/en_PP-OCRv4_rec_hvac.yml \
    -o Global.pretrained_model=./output/rec_ppocr_v4_hvac/best_accuracy \
       Global.save_inference_dir=./inference/rec_ppocr_v4_hvac

In [ ]:
import shutil
shutil.copy('./ocr_finetune/dict.txt', './inference/rec_ppocr_v4_hvac/dict.txt')
shutil.make_archive('/kaggle/working/rec_ppocr_v4_hvac', 'zip', './inference', 'rec_ppocr_v4_hvac')
print('Download /kaggle/working/rec_ppocr_v4_hvac.zip, unzip into models/ in the repo.')